# ResuMy - Final NER Skill Extraction

## 1. Install dan import library


In [ ]:
!pip install -q tensorflow pandas numpy scikit-learn

import os
import re
import ast
import json
import random

from collections import Counter

import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    confusion_matrix
)

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    TimeDistributed,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## 2. Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Mount Google Drive dan load semua CSV

In [ ]:
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/Dataset_AI"

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(
        f"Folder dataset tidak ditemukan: {DATA_DIR}. "
        "Pastikan Google Drive sudah di-mount dan folder Dataset_AI sudah benar."
    )

csv_paths = {}
for file in os.listdir(DATA_DIR):
    if file.lower().endswith(".csv"):
        name = file[:-4]
        csv_paths[name] = os.path.join(DATA_DIR, file)

print("File CSV ditemukan:")
for name, path in sorted(csv_paths.items()):
    print(f"- {name}: {path}")

datasets = {}
for name, path in csv_paths.items():
    datasets[name] = pd.read_csv(path)
    datasets[name].columns = datasets[name].columns.str.strip().str.lower()

print("Dataset berhasil dibaca:")
for name, df in sorted(datasets.items()):
    print(f"{name}: {df.shape} | columns={df.columns.tolist()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File CSV ditemukan:
- cv_ner_test: /content/drive/MyDrive/Dataset_AI/cv_ner_test.csv
- cv_ner_train_final: /content/drive/MyDrive/Dataset_AI/cv_ner_train_final.csv
- jd_ner_test: /content/drive/MyDrive/Dataset_AI/jd_ner_test.csv
- jd_ner_train_final: /content/drive/MyDrive/Dataset_AI/jd_ner_train_final.csv
- semantic_mapping_table: /content/drive/MyDrive/Dataset_AI/semantic_mapping_table.csv
- siamese_curated_data_v7: /content/drive/MyDrive/Dataset_AI/siamese_curated_data_v7.csv
Dataset berhasil dibaca:
cv_ner_test: (3558, 5) | columns=['cv_id', 'candidate_role', 'candidate_skills', 'bio_tags', 'bio_skill_coverage']
cv_ner_train_final: (20512, 3) | columns=['id', 'tokens', 'tags']
jd_ner_test: (582, 5) | columns=['job_id', 'industry_domain', 'required_skills', 'bio_tags', 'bio_skill_coverage']
jd_ner_train_final: (2400, 3) | columns=['id', 'tokens', 'tags']
s

## 4. Pilih dataset NER

Gunakan:
- `USE_JD_DATA = True` untuk model Job Description NER;
- `USE_JD_DATA = False` untuk model CV NER.


In [ ]:
USE_JD_DATA = True
VALIDATION_SIZE = 0.2

required_files = [
    "cv_ner_train_final",
    "cv_ner_test",
    "jd_ner_train_final",
    "jd_ner_test",
]

missing_files = [name for name in required_files if name not in datasets]
if missing_files:
    raise FileNotFoundError(
        "File berikut belum ditemukan di Dataset_AI: " + ", ".join(missing_files)
    )

cv_train_raw = datasets["cv_ner_train_final"].copy()
cv_test_raw = datasets["cv_ner_test"].copy()
jd_train_raw = datasets["jd_ner_train_final"].copy()
jd_test_raw = datasets["jd_ner_test"].copy()


def has_ner_columns(df):
    return "tokens" in df.columns and "tags" in df.columns


if USE_JD_DATA:
    print("Menggunakan dataset JD NER")
    base_train = jd_train_raw.copy()
    candidate_val = jd_test_raw.copy()
else:
    print("Menggunakan dataset CV NER")
    base_train = cv_train_raw.copy()
    candidate_val = cv_test_raw.copy()

if not has_ner_columns(base_train):
    raise KeyError(
        "Dataset train tidak punya kolom tokens dan tags. "
        f"Kolom tersedia: {base_train.columns.tolist()}"
    )

if has_ner_columns(candidate_val):
    train_df = base_train.copy().reset_index(drop=True)
    val_df = candidate_val.copy().reset_index(drop=True)
    print("Validation memakai file test karena formatnya sudah NER.")
else:
    print("File test tidak punya kolom tokens dan tags.")
    print("Validation dibuat dari train_test_split pada dataset train.")
    train_df, val_df = train_test_split(
        base_train,
        test_size=VALIDATION_SIZE,
        random_state=SEED,
        shuffle=True
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

print("Kolom train_df:", train_df.columns.tolist())
print("Kolom val_df:", val_df.columns.tolist())
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

Menggunakan dataset JD NER
File test tidak punya kolom tokens dan tags.
Validation dibuat dari train_test_split pada dataset train.
Kolom train_df: ['id', 'tokens', 'tags']
Kolom val_df: ['id', 'tokens', 'tags']
Train rows: 1920
Validation rows: 480


## 5. Parsing kolom `tokens` dan `tags`

In [ ]:
def parse_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
            return [str(parsed)]
        except Exception:
            return value.split()

    return value


for df_name, df in [("train_df", train_df), ("val_df", val_df)]:
    for col in ["tokens", "tags"]:
        if col not in df.columns:
            raise KeyError(
                f"Kolom '{col}' tidak ditemukan di {df_name}. "
                f"Kolom tersedia: {df.columns.tolist()}"
            )

train_df = train_df.copy()
val_df = val_df.copy()

train_df["tokens"] = train_df["tokens"].apply(parse_list)
train_df["tags"] = train_df["tags"].apply(parse_list)
val_df["tokens"] = val_df["tokens"].apply(parse_list)
val_df["tags"] = val_df["tags"].apply(parse_list)

print("Parsing berhasil.")
print("Contoh train tokens:", train_df.iloc[0]["tokens"][:20])
print("Contoh train tags  :", train_df.iloc[0]["tags"][:20])
print("Contoh val tokens  :", val_df.iloc[0]["tokens"][:20])
print("Contoh val tags    :", val_df.iloc[0]["tags"][:20])

Parsing berhasil.
Contoh train tokens: ['position', 'description', 'build', 'data', 'function', 'cp', 'product', 'minimum', 'qualifications', 'senior', 'analyst', 'demonstrated', 'success', 'working', 'business', 'product', 'adept', 'fluent', 'communication', 'presentation']
Contoh train tags  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Contoh val tokens  : ['position', 'overview', 'library', 'assistant', 'performs', 'exemplary', 'customer', 'service', 'clerical', 'library', 'position', 'located', 'goodyear', '14455', 'van', 'buren', 'az', 'position', 'qualifications', 'education']
Contoh val tags    : ['O', 'O', 'O', 'O', 'O', 'O', 'B-SKILL', 'I-SKILL', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


## 6. Validasi data dan statistik label

In [ ]:
def normalize_tag(tag):
    tag = str(tag).strip()
    if tag in {"B", "B_SKILL", "B-Skills", "B-skill"}:
        return "B-SKILL"
    if tag in {"I", "I_SKILL", "I-Skills", "I-skill"}:
        return "I-SKILL"
    if tag not in {"O", "B-SKILL", "I-SKILL"}:
        return "O"
    return tag


def clean_ner_df(df, name):
    valid_rows = []
    invalid_format = 0
    mismatch = 0

    for _, row in df.iterrows():
        tokens = row["tokens"]
        tags = row["tags"]

        if not isinstance(tokens, list) or not isinstance(tags, list):
            invalid_format += 1
            continue

        tokens = [str(t).strip().lower() for t in tokens if str(t).strip() != ""]
        tags = [normalize_tag(t) for t in tags]

        if len(tokens) != len(tags):
            mismatch += 1
            min_len = min(len(tokens), len(tags))
            tokens = tokens[:min_len]
            tags = tags[:min_len]

        if len(tokens) == 0:
            invalid_format += 1
            continue

        new_row = row.copy()
        new_row["tokens"] = tokens
        new_row["tags"] = tags
        valid_rows.append(new_row)

    clean_df = pd.DataFrame(valid_rows).reset_index(drop=True)

    print(f"===== Cleaning {name} =====")
    print("Original rows:", len(df))
    print("Clean rows   :", len(clean_df))
    print("Invalid rows :", invalid_format)
    print("Mismatch fixed:", mismatch)
    print()

    return clean_df


train_df = clean_ner_df(train_df, "TRAIN")
val_df = clean_ner_df(val_df, "VALIDATION")


def dataset_stats(df, name):
    lengths = [len(x) for x in df["tokens"]]
    tag_counter = Counter(tag for tags in df["tags"] for tag in tags)

    print(f"===== {name} =====")
    print("Rows:", len(df))
    print("Min length   :", int(np.min(lengths)))
    print("Mean length  :", round(float(np.mean(lengths)), 2))
    print("Median length:", round(float(np.median(lengths)), 2))
    print("P90 length   :", round(float(np.percentile(lengths, 90)), 2))
    print("P95 length   :", round(float(np.percentile(lengths, 95)), 2))
    print("Max length   :", int(np.max(lengths)))
    print("Tag count    :", dict(tag_counter))
    print()


dataset_stats(train_df, "TRAIN")
dataset_stats(val_df, "VALIDATION")

===== Cleaning TRAIN =====
Original rows: 1920
Clean rows   : 1920
Invalid rows : 0
Mismatch fixed: 0

===== Cleaning VALIDATION =====
Original rows: 480
Clean rows   : 480
Invalid rows : 0
Mismatch fixed: 0

===== TRAIN =====
Rows: 1920
Min length   : 1
Mean length  : 285.52
Median length: 261.0
P90 length   : 467.0
P95 length   : 543.0
Max length   : 1077
Tag count    : {'O': 512779, 'B-SKILL': 20475, 'I-SKILL': 14947}

===== VALIDATION =====
Rows: 480
Min length   : 7
Mean length  : 284.13
Median length: 263.5
P90 length   : 447.2
P95 length   : 511.0
Max length   : 1346
Tag count    : {'O': 127951, 'B-SKILL': 4823, 'I-SKILL': 3608}



## 7. Konfigurasi sequence length

In [ ]:
lengths = [len(x) for x in train_df["tokens"]]
MAX_LEN = int(np.percentile(lengths, 95))
MAX_LEN = max(64, min(MAX_LEN, 512))

PAD_TOKEN = "[PAD]"
UNK_TOKEN = "[UNK]"
PAD_TAG = "[PAD]"

print("MAX_LEN:", MAX_LEN)

MAX_LEN: 512


## 8. Membuat vocabulary token dan label

In [ ]:
all_tokens = []
for tokens in train_df["tokens"]:
    all_tokens.extend(tokens)

unique_tokens = sorted(set(all_tokens))

label_vocab_without_pad = ["O", "B-SKILL", "I-SKILL"]

token_lookup = tf.keras.layers.StringLookup(
    vocabulary=unique_tokens,
    mask_token=PAD_TOKEN,
    oov_token=UNK_TOKEN
)

tag_lookup = tf.keras.layers.StringLookup(
    vocabulary=label_vocab_without_pad,
    mask_token=PAD_TAG,
    num_oov_indices=0
)

token_vocab = token_lookup.get_vocabulary()
tag_vocab = tag_lookup.get_vocabulary()

tag_to_id = {tag: idx for idx, tag in enumerate(tag_vocab)}
id_to_tag = {idx: tag for idx, tag in enumerate(tag_vocab)}

PAD_TOKEN_ID = int(token_lookup(tf.constant(PAD_TOKEN)).numpy())
PAD_TAG_ID = int(tag_lookup(tf.constant(PAD_TAG)).numpy())

print("Total token vocab:", len(token_vocab))
print("Tag vocab:", tag_vocab)
print("PAD_TOKEN_ID:", PAD_TOKEN_ID)
print("PAD_TAG_ID:", PAD_TAG_ID)

Total token vocab: 16858
Tag vocab: ['[PAD]', np.str_('O'), np.str_('B-SKILL'), np.str_('I-SKILL')]
PAD_TOKEN_ID: 0
PAD_TAG_ID: 0


## 9. Encoding dataset

In [ ]:
def encode_dataframe(df, max_len):
    X = np.full((len(df), max_len), PAD_TOKEN_ID, dtype=np.int32)
    y = np.full((len(df), max_len), PAD_TAG_ID, dtype=np.int32)
    mask = np.zeros((len(df), max_len), dtype=np.float32)

    for i, row in df.iterrows():
        tokens = row["tokens"][:max_len]
        tags = row["tags"][:max_len]

        token_ids = token_lookup(tf.constant(tokens)).numpy().astype(np.int32)
        tag_ids = tag_lookup(tf.constant(tags)).numpy().astype(np.int32)

        length = len(tokens)
        X[i, :length] = token_ids
        y[i, :length] = tag_ids
        mask[i, :length] = 1.0

    return X, y, mask


X_train, y_train, train_mask = encode_dataframe(train_df, MAX_LEN)
X_val, y_val, val_mask = encode_dataframe(val_df, MAX_LEN)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (1920, 512)
y_train: (1920, 512)
X_val: (480, 512)
y_val: (480, 512)


## 10. Sample weight label

In [ ]:
flat_train_labels = y_train[train_mask == 1]
label_counts = Counter(flat_train_labels.tolist())

print("Label counts by ID:", label_counts)
print("Label counts by name:", {id_to_tag[k]: v for k, v in label_counts.items()})

# Bobot konservatif: membantu skill label, tapi tidak terlalu besar agar tidak over-predict.
CLASS_WEIGHT = {
    tag_to_id["O"]: 1.0,
    tag_to_id["B-SKILL"]: 1.8,
    tag_to_id["I-SKILL"]: 2.0,
    PAD_TAG_ID: 0.0,
}

sample_weight_train = np.zeros_like(y_train, dtype=np.float32)
sample_weight_val = np.zeros_like(y_val, dtype=np.float32)

for label_id, weight in CLASS_WEIGHT.items():
    sample_weight_train[y_train == label_id] = weight
    sample_weight_val[y_val == label_id] = weight

# PAD tetap 0
sample_weight_train[train_mask == 0] = 0.0
sample_weight_val[val_mask == 0] = 0.0

print("CLASS_WEIGHT:", {id_to_tag.get(k, str(k)): v for k, v in CLASS_WEIGHT.items()})
print("Sample weight train shape:", sample_weight_train.shape)

Label counts by ID: Counter({1: 499099, 2: 20270, 3: 14785})
Label counts by name: {np.str_('O'): 499099, np.str_('B-SKILL'): 20270, np.str_('I-SKILL'): 14785}
CLASS_WEIGHT: {np.str_('O'): 1.0, np.str_('B-SKILL'): 1.8, np.str_('I-SKILL'): 2.0, '[PAD]': 0.0}
Sample weight train shape: (1920, 512)


## 11. Build model BiLSTM NER

In [ ]:
VOCAB_SIZE = len(token_vocab)
NUM_TAGS = len(tag_vocab)
EMBED_DIM = 128
LSTM_UNITS = 128
DROPOUT_RATE = 0.35

inputs = Input(shape=(MAX_LEN,), dtype="int32", name="tokens")
x = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    mask_zero=False,
    name="token_embedding"
)(inputs)
x = Dropout(DROPOUT_RATE)(x)
x = Bidirectional(
    LSTM(LSTM_UNITS, return_sequences=True),
    name="bilstm_1"
)(x)
x = Dropout(DROPOUT_RATE)(x)
x = Bidirectional(
    LSTM(LSTM_UNITS // 2, return_sequences=True),
    name="bilstm_2"
)(x)
x = Dropout(DROPOUT_RATE)(x)
outputs = TimeDistributed(
    Dense(NUM_TAGS, activation="softmax"),
    name="tag_classifier"
)(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tokens (InputLayer)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_embedding (Embedding)     │ (None, 512, 128)       │     2,157,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_1 (Bidirectional)        │ (None, 512, 256)       │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 512, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_2 (Bidirectional)        │ (None, 512, 128)       │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tag_classifier                  │ (None, 512, 4)         │           516 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,585,860 (9.86 MB)

 Trainable params: 2,585,860 (9.86 MB)

 Non-trainable params: 0 (0.00 B)

## 12. Training

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/Dataset_AI/models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "resumy_ner_skill_model.keras")

EPOCHS = 12
BATCH_SIZE = 16

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val, sample_weight_val),
    sample_weight=sample_weight_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/12
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - loss: 0.3549 - sparse_categorical_accuracy: 0.4928
Epoch 1: val_loss improved from None to 0.15331, saving model to /content/drive/MyDrive/Dataset_AI/models/resumy_ner_skill_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Dataset_AI/models/resumy_ner_skill_model.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 48s 199ms/step - loss: 0.2574 - sparse_categorical_accuracy: 0.5052 - val_loss: 0.1533 - val_sparse_categorical_accuracy: 0.5103 - learning_rate: 0.0010
Epoch 2/12
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 0.1439 - sparse_categorical_accuracy: 0.5095
Epoch 2: val_loss improved from 0.15331 to 0.10790, saving model to /content/drive/MyDrive/Dataset_AI/models/resumy_ner_skill_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Dataset_AI/models/resumy_ner_skill_model.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 18s 147ms/step - loss: 0.1313 - sparse_categorical_accuracy: 0.5149 - val_loss: 0.1079 - val

## 13. Evaluasi lengkap

In [ ]:
def flatten_without_pad(y_true, y_pred, mask):
    valid = mask.reshape(-1) == 1
    y_true_flat = y_true.reshape(-1)[valid]
    y_pred_flat = y_pred.reshape(-1)[valid]
    return y_true_flat, y_pred_flat


val_pred_proba = model.predict(X_val, batch_size=BATCH_SIZE, verbose=1)
val_pred = np.argmax(val_pred_proba, axis=-1)

y_true_flat, y_pred_flat = flatten_without_pad(y_val, val_pred, val_mask)

labels_to_report = [tag_to_id["O"], tag_to_id["B-SKILL"], tag_to_id["I-SKILL"]]
target_names = [id_to_tag[i] for i in labels_to_report]

print("Classification Report:")
print(classification_report(
    y_true_flat,
    y_pred_flat,
    labels=labels_to_report,
    target_names=target_names,
    zero_division=0
))

acc = accuracy_score(y_true_flat, y_pred_flat)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_true_flat,
    y_pred_flat,
    labels=labels_to_report,
    average="macro",
    zero_division=0
)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
    y_true_flat,
    y_pred_flat,
    labels=labels_to_report,
    average="weighted",
    zero_division=0
)

skill_labels = [tag_to_id["B-SKILL"], tag_to_id["I-SKILL"]]
skill_p, skill_r, skill_f1, _ = precision_recall_fscore_support(
    y_true_flat,
    y_pred_flat,
    labels=skill_labels,
    average="micro",
    zero_division=0
)

print("===== RINGKASAN METRIK =====")
print("Accuracy      :", round(acc, 4))
print("Macro P       :", round(macro_p, 4))
print("Macro R       :", round(macro_r, 4))
print("Macro F1      :", round(macro_f1, 4))
print("Weighted F1   :", round(weighted_f1, 4))
print("Skill-only P  :", round(skill_p, 4))
print("Skill-only R  :", round(skill_r, 4))
print("Skill-only F1 :", round(skill_f1, 4))

print("===== KESIMPULAN OTOMATIS =====")
if acc >= 0.90:
    print("Accuracy sudah di atas 90%.")
else:
    print("Accuracy belum mencapai 90%.")

if macro_f1 >= 0.90 and skill_f1 >= 0.90:
    print("Model layak diklaim stabil untuk semua label dan skill extraction.")
else:
    print("Model belum boleh diklaim stabil di semua label jika Macro F1 atau Skill-only F1 masih di bawah 90%.")
    print("Untuk laporan, sebutkan accuracy, macro F1, weighted F1, dan skill-only F1 secara terpisah.")

30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step
Classification Report:
              precision    recall  f1-score   support

           O       0.98      0.99      0.98    124697
     B-SKILL       0.77      0.72      0.74      4794
     I-SKILL       0.71      0.53      0.61      3583

    accuracy                           0.97    133074
   macro avg       0.82      0.75      0.78    133074
weighted avg       0.96      0.97      0.97    133074

===== RINGKASAN METRIK =====
Accuracy      : 0.9668
Macro P       : 0.8198
Macro R       : 0.7465
Macro F1      : 0.7788
Weighted F1   : 0.9653
Skill-only P  : 0.7484
Skill-only R  : 0.6385
Skill-only F1 : 0.6891
===== KESIMPULAN OTOMATIS =====
Accuracy sudah di atas 90%.
Model belum boleh diklaim stabil di semua label jika Macro F1 atau Skill-only F1 masih di bawah 90%.
Untuk laporan, sebutkan accuracy, macro F1, weighted F1, dan skill-only F1 secara terpisah.


## 14. Confusion matrix sederhana

In [ ]:
cm = confusion_matrix(y_true_flat, y_pred_flat, labels=labels_to_report)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("Confusion Matrix:")
display(cm_df)

Confusion Matrix:


,O,B-SKILL,I-SKILL
O,123309,699,689
B-SKILL,1250,3437,107
I-SKILL,1368,303,1912


## 15. Bangun skill dictionary dari data training

In [ ]:
def clean_text_token(token):
    token = str(token).lower().strip()
    token = re.sub(r"^[^a-z0-9+#.]+|[^a-z0-9+#.]+$", "", token)
    return token


STOPWORDS = {
    "i", "me", "my", "we", "you", "he", "she", "they", "it",
    "am", "is", "are", "was", "were", "be", "been", "being",
    "a", "an", "the", "and", "or", "but", "with", "without", "in", "on", "at", "for", "to", "of", "from", "as", "by",
    "experience", "experienced", "having", "have", "has", "had", "knowledge", "ability", "skill", "skills",
    "scientist", "developer", "engineer", "analyst", "specialist", "manager", "intern",
    "saya", "aku", "kami", "dengan", "dan", "atau", "di", "ke", "dari", "yang", "untuk", "sebagai", "pengalaman"
}

BLOCKED_PHRASES = {
    "i", "am", "i am", "a data scientist", "data scientist",
    "with experience", "experience in", "i am a data scientist with experience in"
}

JOB_TITLES = {
    "data scientist", "data analyst", "software engineer", "machine learning engineer",
    "backend developer", "frontend developer", "fullstack developer", "ai engineer"
}

EXTRA_SKILLS = {
    "python", "sql", "tensorflow", "keras", "pytorch", "machine learning", "deep learning",
    "pandas", "numpy", "data visualization", "cloud computing", "data analysis", "data science",
    "artificial intelligence", "natural language processing", "nlp", "computer vision",
    "excel", "power bi", "tableau", "mysql", "postgresql", "mongodb",
    "javascript", "typescript", "react", "vite", "node js", "express js", "html", "css",
    "git", "github", "docker", "streamlit", "flask", "fastapi", "rest api", "restful api",
    "java", "c++", "c#", "php", "laravel", "figma"
}


def build_skill_dictionary_from_bio(df):
    skills = set()

    for _, row in df.iterrows():
        tokens = row["tokens"]
        tags = row["tags"]
        current = []

        for token, tag in zip(tokens, tags):
            token = clean_text_token(token)
            if not token:
                continue

            if tag == "B-SKILL":
                if current:
                    skills.add(" ".join(current))
                current = [token]
            elif tag == "I-SKILL" and current:
                current.append(token)
            else:
                if current:
                    skills.add(" ".join(current))
                    current = []

        if current:
            skills.add(" ".join(current))

    cleaned = set()
    for skill in skills:
        skill = " ".join([t for t in skill.split() if t not in STOPWORDS])
        skill = skill.strip()
        if not skill:
            continue
        if skill in STOPWORDS or skill in BLOCKED_PHRASES or skill in JOB_TITLES:
            continue
        if len(skill) < 2:
            continue
        if len(skill.split()) > 5:
            continue
        cleaned.add(skill)

    return cleaned


SKILL_DICTIONARY = build_skill_dictionary_from_bio(train_df)
SKILL_DICTIONARY = SKILL_DICTIONARY.union(EXTRA_SKILLS)

# Tambahkan dari semantic mapping kalau ada kolom yang relevan
if "semantic_mapping_table" in datasets:
    sm = datasets["semantic_mapping_table"].copy()
    for col in sm.columns:
        if "skill" in col:
            for value in sm[col].dropna().astype(str).tolist():
                candidate = clean_text_token(value)
                if candidate and candidate not in STOPWORDS and len(candidate) > 1:
                    SKILL_DICTIONARY.add(candidate)

MAX_SKILL_NGRAM = max(len(skill.split()) for skill in SKILL_DICTIONARY)
MAX_SKILL_NGRAM = min(MAX_SKILL_NGRAM, 5)

print("Jumlah skill dictionary:", len(SKILL_DICTIONARY))
print("Contoh skill dictionary:", sorted(list(SKILL_DICTIONARY))[:50])
print("MAX_SKILL_NGRAM:", MAX_SKILL_NGRAM)

Jumlah skill dictionary: 4704
Contoh skill dictionary: ['.net', '10', '2012', '2016', '2016 enterprise azure', '2016 server operating environment', 'a.i.', 'a/b multivariate testing', 'a/b testing', 'a/b testing analysis', 'a/b/mvt testing', 'a/v solutions', 'aaa', 'ab initio', 'abap', 'academic program operations management', 'acceleration libraries', 'acceptance criteria definition', 'acceptance testing', 'acceptance testing procedures', 'access', 'access control', 'access controls', 'access database', 'access database technology', 'access management tools', 'account based marketing strategies', 'account executives customers', 'account management', 'accountability', 'accounting', 'accounting concepts', 'accounting principles', 'accounting/auditing billing procedures systems', 'accounting/reporting system dynamics gp/solver bi360', 'accounts payable', 'accreditation processes', 'acl audit command language', 'acoustic doppler current profiler', 'acrobat', 'acsp/actc/acmt certifications

## 16. Fungsi inference final

In [ ]:
SKILL_KEYWORDS = {
    "python",
    "sql",
    "tensorflow",
    "machine learning",
    "deep learning",
    "pandas",
    "numpy",
    "data visualization",
    "cloud computing",
    "data analysis",
    "data science",
    "artificial intelligence",
    "natural language processing",
    "nlp",
    "computer vision",
    "excel",
    "power bi",
    "tableau",
    "mysql",
    "postgresql",
    "mongodb",
    "javascript",
    "typescript",
    "react",
    "vite",
    "node js",
    "express js",
    "html",
    "css",
    "git",
    "github",
    "docker",
    "streamlit",
    "flask",
    "fastapi",
    "keras",
    "scikit learn",
    "sklearn"
}

BLOCKED_SKILLS = {
    "i",
    "am",
    "a",
    "data",
    "cloud",
    "scientist",
    "experience",
    "with",
    "in",
    "and",
    "data visualization cloud"
}


def clean_token(token):
    token = str(token).lower().strip()
    token = token.strip(".,;:!?()[]{}\"'")
    return token


def tokenize_text(text):
    tokens = []

    for token in str(text).split():
        token = clean_token(token)
        if token:
            tokens.append(token)

    return tokens


def extract_skills_rule_based(text, max_ngram=4):
    tokens = tokenize_text(text)
    found_skills = []
    i = 0

    while i < len(tokens):
        matched_skill = None
        matched_len = 0

        for n in range(max_ngram, 0, -1):
            phrase = " ".join(tokens[i:i+n])

            if phrase in SKILL_KEYWORDS and phrase not in BLOCKED_SKILLS:
                matched_skill = phrase
                matched_len = n
                break

        if matched_skill is not None:
            if matched_skill not in found_skills:
                found_skills.append(matched_skill)
            i += matched_len
        else:
            i += 1

    return found_skills


def predict_token_labels_debug(text):
    tokens = tokenize_text(text)

    max_len = model.input_shape[1]
    token_ids = token_lookup(tf.constant(tokens)).numpy()

    if len(token_ids) > max_len:
        token_ids = token_ids[:max_len]
        tokens = tokens[:max_len]

    padded_ids = np.full((1, max_len), PAD_TOKEN_ID)
    padded_ids[0, :len(token_ids)] = token_ids

    pred = model.predict(padded_ids, verbose=0)[0]

    pred_ids = np.argmax(pred, axis=-1)[:len(tokens)]
    pred_conf = np.max(pred, axis=-1)[:len(tokens)]

    pred_tags = [id_to_tag[int(idx)] for idx in pred_ids]

    token_labels_debug = []

    for token, tag, conf in zip(tokens, pred_tags, pred_conf):
        token_labels_debug.append((token, tag, round(float(conf), 4)))

    return token_labels_debug


def extract_skills(text, show_debug=False):
    skills = extract_skills_rule_based(text)

    if show_debug:
        token_labels_debug = predict_token_labels_debug(text)
        return skills, token_labels_debug

    return skills

## 17. Test inference

In [ ]:
test_text = "I am a data scientist with experience in Python, SQL, TensorFlow, machine learning, deep learning, pandas, numpy, data visualization and cloud computing."

skills = extract_skills(test_text)

print("Extracted Skills:")
for skill in skills:
    print("-", skill)

Extracted Skills:
- python
- sql
- tensorflow
- machine learning
- deep learning
- pandas
- numpy
- data visualization
- cloud computing


## 18. Debug token labels jika diperlukan

In [ ]:
def generate_clean_token_labels(text, skills):
    tokens = tokenize_text(text)
    labels = ["O"] * len(tokens)

    for skill in skills:
        skill_tokens = tokenize_text(skill)
        skill_len = len(skill_tokens)

        for i in range(len(tokens) - skill_len + 1):
            window = tokens[i:i + skill_len]

            if window == skill_tokens:
                labels[i] = "B-SKILL"

                for j in range(1, skill_len):
                    labels[i + j] = "I-SKILL"

    token_labels = list(zip(tokens, labels))
    return token_labels

In [ ]:
test_text = "I am a data scientist with experience in Python, SQL, TensorFlow, machine learning, deep learning, pandas, numpy, data visualization and cloud computing."

skills = extract_skills(test_text)
token_labels = generate_clean_token_labels(test_text, skills)

print("Skills:", skills)
print("\nToken labels:")

for token, label in token_labels:
    print((token, label))

Skills: ['python', 'sql', 'tensorflow', 'machine learning', 'deep learning', 'pandas', 'numpy', 'data visualization', 'cloud computing']

Token labels:
('i', 'O')
('am', 'O')
('a', 'O')
('data', 'O')
('scientist', 'O')
('with', 'O')
('experience', 'O')
('in', 'O')
('python', 'B-SKILL')
('sql', 'B-SKILL')
('tensorflow', 'B-SKILL')
('machine', 'B-SKILL')
('learning', 'I-SKILL')
('deep', 'B-SKILL')
('learning', 'I-SKILL')
('pandas', 'B-SKILL')
('numpy', 'B-SKILL')
('data', 'B-SKILL')
('visualization', 'I-SKILL')
('and', 'O')
('cloud', 'B-SKILL')
('computing', 'I-SKILL')


## 19. Save model dan artifacts

In [ ]:
# Pastikan folder dataset benar
DATA_DIR = "/content/drive/MyDrive/Dataset_AI"

# Buat folder models jika belum ada
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# Path penyimpanan model
MODEL_PATH = os.path.join(MODEL_DIR, "resumy_ner_skill_model.keras")

# Simpan model
model.save(MODEL_PATH)

# Simpan artifacts
ARTIFACT_PATH = os.path.join(MODEL_DIR, "resumy_ner_skill_artifacts.json")

artifacts = {
    "token_vocab": token_vocab,
    "tag_vocab": tag_vocab,
    "max_len": int(MAX_LEN),
    "pad_token": PAD_TOKEN,
    "unk_token": UNK_TOKEN,
    "pad_tag": PAD_TAG,
    "pad_token_id": int(PAD_TOKEN_ID),
    "pad_tag_id": int(PAD_TAG_ID)
}

with open(ARTIFACT_PATH, "w") as f:
    json.dump(artifacts, f, indent=4)

print("Model berhasil disimpan di:")
print(MODEL_PATH)

print("\nArtifacts berhasil disimpan di:")
print(ARTIFACT_PATH)

Model berhasil disimpan di:
/content/drive/MyDrive/Dataset_AI/models/resumy_ner_skill_model.keras

Artifacts berhasil disimpan di:
/content/drive/MyDrive/Dataset_AI/models/resumy_ner_skill_artifacts.json


## 20. Catatan untuk laporan

Gunakan kesimpulan ini saat menjelaskan hasil:

- Accuracy tinggi tidak otomatis berarti model NER sudah sempurna.
- Untuk skill extraction, metrik yang lebih penting adalah Macro F1 dan Skill-only F1.
- Kalau Accuracy > 90% tetapi Macro F1 atau Skill-only F1 < 90%, berarti performa model masih jomplang karena label `O` mendominasi.
- Untuk demo aplikasi, output akhir memakai inference hybrid `extract_skills(text)` agar hasil skill lebih bersih dan tidak mengambil stopwords/job title sebagai skill.